# 📜 Citadel Access Contracts - Testing Center

## Overview

Provision and validate **multiple Access Contracts** with different configurations and targets, each
backed by a **dynamically generated APIM product policy** that enforces:

- **Model-level RBAC** via the `allowedModels` variable (per-contract list of allowed model names)
- **Capacity allocation** via `llm-token-limit` (`tokens-per-minute`, `token-quota`, `token-quota-period`)
- **Optional Azure Key Vault integration** (resolve endpoint + API key secrets)
- **Optional Microsoft Foundry connection integration** (auto-create a Foundry project connection)

| Access Contract | Configuration | Description | Target Runtime |
|---|---|---|---|
| Sales-Assistant | Key Vault integration only | Sales Assistant workload with secrets resolved from Key Vault. | Agents running on Azure (AKS, ACA, App Service) |
| HR-ChatAgent | Key Vault + Foundry connection integrations | HR chat agent flow with Foundry connection setup. | Foundry Agents |
| Support-Bot | Direct output (no Key Vault nor Foundry integration) | Support bot scenario using direct output without external integrations. | Custom hosted agents |

## Expected outcomes

- Three Access Contracts (APIM products + subscriptions) deployed via Bicep
- A per-contract APIM product policy (`ai-product-policy.xml`) generated **dynamically** from
  the notebook variables (model RBAC + capacity)
- Foundry connection name(s) printed explicitly for use in downstream notebooks
  (e.g. `citadel-agent-frameworks-tests.ipynb`)
- Validated request/response behaviour, throttling and token-bucket visualisation

## Azure Prerequisites

To take full advantage of this notebook, ensure you have the following Azure resources set up:
- An existing Citadel Governance Hub deployment (APIM + supporting resources)
- *(Optional)* An Azure Key Vault with secrets for API keys (if testing Key Vault integration)
- *(Optional)* A Microsoft Foundry account & project (if testing Foundry connection integration)
- Azure credentials with permissions to deploy at subscription scope and access the above resources

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Access Contracts Guide](../guides/full-deployment-guide.md) before proceeding.


<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Start by defining the variables that will drive the configuration of your Access Contracts and the generation of the APIM product policies. Adjust these variables according to your testing needs.


In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool


# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults)
# ============================================================================
governance_hub_resource_group = ""   # Resource group of the deployed Citadel Governance Hub
location = ""                         # Azure region (e.g. "swedencentral", "eastus")

# ============================================================================
# 🔧 API VERSION CONFIGURATION
# ============================================================================
inference_api_version = "2024-05-01-preview"
openai_api_version    = "2024-12-01-preview"
targetInferenceApi    = "models"             # 'models' = Universal LLM API | 'openai' = Azure OpenAI API

# ============================================================================
# 🔐 KEY VAULT CONFIGURATION (optional - set use_keyvault_integration = True to enable)
# ----------------------------------------------------------------------------
# Any value left as "REPLACE" (or empty) will be filled in from azd when
# `init_from_azd = True`. Any value you set here explicitly will be honored
# as-is and will NOT be overridden by azd. This lets you point the contract
# at a different Key Vault (e.g. a shared central vault) than the one azd
# provisioned with the hub.
# ============================================================================
use_keyvault_integration = False
keyvault_subscription_id = "REPLACE"
keyvault_resource_group  = "REPLACE"
keyvault_name            = "REPLACE"

# ============================================================================
# 🤖 MICROSOFT FOUNDRY CONFIGURATION (optional - set use_foundry_integration = True to enable)
# Same precedence rule as Key Vault: explicit values win, "REPLACE" is filled by azd.
# ============================================================================
use_foundry_integration = False
foundry_subscription_id = "REPLACE"
foundry_resource_group  = "REPLACE"
foundry_account_name    = "REPLACE"
foundry_project_name    = "REPLACE"


utils.print_ok("Notebook variables initialized!")


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [ ]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    apimClientTool.discover_api(targetInferenceApi)

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)
    
    # Get supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models in APIM policy fragment 'set-backend-pools': {supported_models}")

    if targetInferenceApi == "openai":
        chat_completions_url = f"{azure_endpoint}openai/deployments/{{model_name}}/chat/completions?api-version={inference_api_version}"
    else:  # models
        chat_completions_url = f"{azure_endpoint}models/chat/completions?api-version={inference_api_version}"
    utils.print_info(f"Chat Completion Endpoint Template: {chat_completions_url}")

    utils.print_info(f"Using the following API: {apimClientTool.api_id}")

    utils.print_ok(f"Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

<a id='3'></a>
### 3️⃣ Define Access Contract Configurations

We will create 3 different access contracts with varying default configurations:
1. **Sales-Assistant**: Key Vault integration only
2. **HR-ChatAgent**: Key Vault + Foundry connection integrations (if enabled)
3. **Support-Bot**: Direct output (no Key Vault nor Foundry integration)

In [ ]:
# Define the access contracts to create
# Each contract supports per-contract:
#   • allowed_models: list of model names this contract is allowed to call (model-level RBAC)
#   • tokens_per_minute: TPM throttle limit applied to the contract
#   • token_quota / token_quota_period: long-term token budget (Hourly | Daily | Weekly | Monthly)
# These values are injected into a dynamically-generated APIM product policy
# (see next cell) so each contract gets its own scoped policy XML.

timestamp = time.strftime('%Y%m%d%H%M%S')

access_contracts = [
    {
        "name": f"sales-assistant-contract-{timestamp}",
        "business_unit": "Sales",
        "use_case_name": "Assistant",
        "environment": "DEV",
        "use_keyvault": False,
        "use_foundry": False,
        "endpoint_secret": "SALES-LLM-ENDPOINT",
        "apikey_secret": "SALES-LLM-KEY",
        "description": "Sales Assistant - Key Vault only",
        # --- Model RBAC (allowed model names, comma-joined into the APIM policy)
        "allowed_models": ["gpt-4.1", "gpt-5.4", "text-embedding-3-small"],
        # --- Capacity allocation (llm-token-limit policy)
        "tokens_per_minute": 3000,
        "token_quota": 100000,
        "token_quota_period": "Monthly"
    },
    {
        "name": f"hr-chatagent-contract-{timestamp}",
        "business_unit": "HR",
        "use_case_name": "ChatAgent",
        "environment": "DEV",
        "use_keyvault": False,
        "use_foundry": False,
        "endpoint_secret": "HR-LLM-ENDPOINT",
        "apikey_secret": "HR-LLM-KEY",
        "description": "HR Chat Agent - Key Vault + Foundry (if enabled)",
        "allowed_models": ["gpt-4.1", "gpt-4.1-mini"],
        "tokens_per_minute": 5000,
        "token_quota": 500000,
        "token_quota_period": "Monthly"
    },
    {
        "name": f"support-bot-contract-{timestamp}",
        "business_unit": "Support",
        "use_case_name": "Bot",
        "environment": "DEV",
        "use_keyvault": False,
        "use_foundry": False,
        "endpoint_secret": "SUPPORT-LLM-ENDPOINT",
        "apikey_secret": "SUPPORT-LLM-KEY",
        "description": "Support Bot - Direct output (no Key Vault nor Foundry connection integration)",
        "allowed_models": ["gpt-4.1"],
        "tokens_per_minute": 2000,
        "token_quota": 50000,
        "token_quota_period": "Daily"
    }
]

utils.print_info(f"Defined {len(access_contracts)} access contracts to create:")
for i, contract in enumerate(access_contracts, 1):
    utils.print_info(f"  {i}. {contract['description']}")
    utils.print_info(f"     Product ID: LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}")
    utils.print_info(f"     Allowed models: {', '.join(contract['allowed_models'])}")
    utils.print_info(f"     Capacity: {contract['tokens_per_minute']} TPM, quota {contract['token_quota']} / {contract['token_quota_period']}")


<a id='4'></a>
### 4️⃣ Create Access Contract Terraform Variables

Generate Terraform variable files (`.tfvars`) for each access contract, targeting the standalone
`citadel-access-contracts` Terraform module. Each file configures the APIM products, subscriptions,
and optionally Key Vault secrets and Foundry connections. The per-contract dynamic policy (model
RBAC + capacity) is embedded directly in the `services[].policy_xml` value as an HCL heredoc.

In [ ]:
# Path to the standalone Terraform module (relative to this notebook in validation/)
tf_dir = os.path.abspath("../citadel-access-contracts")
if not os.path.isdir(tf_dir):
    utils.print_error(f"Terraform module directory not found: {tf_dir}")

# Generated per-contract tfvars files live under the module's contracts/ folder
tfvars_out_dir = os.path.join(tf_dir, "contracts")
os.makedirs(tfvars_out_dir, exist_ok=True)

# Store generated tfvars files for deployment
generated_tfvars_files = []


def hcl_str(value):
    """Render a Python value as a double-quoted HCL string literal."""
    return '"' + str(value).replace("\\", "\\\\").replace('"', '\\"') + '"'


def build_product_policy_xml(contract):
    """Generate a per-contract APIM product policy XML applying:
       - Model-level RBAC via the `allowedModels` variable + validate-model-access fragment
       - Capacity allocation via llm-token-limit (tokens-per-minute + token-quota)
    Mirrors the dynamic-policy approach used in citadel-unified-ai-api-tests.ipynb."""
    allowed_csv = ",".join(contract["allowed_models"])
    return f'''<policies>
    <inbound>
        <base />
        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Model-level RBAC: only the models below are allowed for this contract -->
        <set-variable name="allowedModels" value="{allowed_csv}" />
        <include-fragment fragment-id="validate-model-access" />

        <!-- Capacity allocation: per-subscription token throttling + long-term quota -->
        <llm-token-limit counter-key="@(context.Subscription.Id)"
                         tokens-per-minute="{contract["tokens_per_minute"]}"
                         estimate-prompt-tokens="false"
                         token-quota="{contract["token_quota"]}"
                         token-quota-period="{contract["token_quota_period"]}" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''


def foundry_static_models_hcl():
    """Render the static models list (Bicep parity) as an HCL list of objects."""
    models = [
        {"name": "gpt-4.1", "version": "2025-04-14", "format": "OpenAI"},
        {"name": "gpt-5.4-mini", "version": "2025-04-14", "format": "OpenAI"},
    ]
    items = []
    for m in models:
        items.append(
            "    {\n"
            "      name = " + hcl_str(m["name"]) + "\n"
            "      properties = {\n"
            "        model = {\n"
            "          name    = " + hcl_str(m["name"]) + "\n"
            "          version = " + hcl_str(m["version"]) + "\n"
            "          format  = " + hcl_str(m["format"]) + "\n"
            "        }\n"
            "      }\n"
            "    }"
        )
    return "[\n" + ",\n".join(items) + "\n  ]"


def build_tfvars(contract, policy_xml):
    """Assemble the full .tfvars content for one access contract (HCL via concatenation
    to avoid f-string brace pitfalls)."""
    # Key Vault block
    if contract["use_keyvault"]:
        kv_block = (
            "use_target_key_vault = true\n\n"
            "key_vault = {\n"
            "  subscription_id     = " + hcl_str(keyvault_subscription_id) + "\n"
            "  resource_group_name = " + hcl_str(keyvault_resource_group) + "\n"
            "  name                = " + hcl_str(keyvault_name) + "\n"
            "}\n"
        )
    else:
        kv_block = "use_target_key_vault = false\n"

    # Foundry block
    if contract["use_foundry"]:
        foundry_block = (
            "\nuse_target_foundry = true\n\n"
            "foundry = {\n"
            "  subscription_id     = " + hcl_str(foundry_subscription_id) + "\n"
            "  resource_group_name = " + hcl_str(foundry_resource_group) + "\n"
            "  account_name        = " + hcl_str(foundry_account_name) + "\n"
            "  project_name        = " + hcl_str(foundry_project_name) + "\n"
            "}\n\n"
            "foundry_config = {\n"
            "  connection_name_prefix = \"\"\n"
            "  connection_category    = \"ApiManagement\"\n"
            "  deployment_in_path     = \"false\"\n"
            "  is_shared_to_all       = false\n"
            "  inference_api_version  = \"2024-05-01-preview\"\n"
            "  deployment_api_version = \"\"\n"
            "  static_models          = " + foundry_static_models_hcl() + "\n"
            "  list_models_endpoint   = \"\"\n"
            "  get_model_endpoint     = \"\"\n"
            "  deployment_provider    = \"\"\n"
            "  custom_headers         = {}\n"
            "  auth_config            = {}\n"
            "}\n"
        )
    else:
        foundry_block = "\nuse_target_foundry = false\n"

    allowed_csv = ",".join(contract["allowed_models"])
    product_terms = f"Access Contract created from testing notebook - {contract['description']}"

    content = (
        "# ============================================================================\n"
        f"# Citadel Access Contract - Generated Terraform Variables\n"
        f"# Contract:  {contract['description']}\n"
        f"# Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"# Dynamic policy: allowedModels={allowed_csv} | TPM={contract['tokens_per_minute']}"
        f" | quota={contract['token_quota']}/{contract['token_quota_period']}\n"
        "# ============================================================================\n\n"
        "# Target APIM (provider is pinned to apim.subscription_id)\n"
        "apim = {\n"
        "  subscription_id     = " + hcl_str(subscription_id) + "\n"
        "  resource_group_name = " + hcl_str(governance_hub_resource_group) + "\n"
        "  name                = " + hcl_str(apimClientTool.apim_resource_name) + "\n"
        "}\n\n"
        "# Use-case identity -> naming: <code>-<business_unit>-<use_case_name>-<environment>\n"
        "use_case = {\n"
        "  business_unit = " + hcl_str(contract["business_unit"]) + "\n"
        "  use_case_name = " + hcl_str(contract["use_case_name"]) + "\n"
        "  environment   = " + hcl_str(contract["environment"]) + "\n"
        "}\n\n"
        "# Existing APIM APIs bundled into each product\n"
        "api_name_mapping = {\n"
        "  LLM = [\"universal-llm-api\", \"azure-openai-api\", \"unified-ai-api\"]\n"
        "}\n\n"
        "# Key Vault secret storage\n"
        + kv_block +
        "\n# Services to onboard (per-contract dynamic policy: model RBAC + capacity)\n"
        "services = [\n"
        "  {\n"
        "    code                 = \"LLM\"\n"
        "    endpoint_secret_name = " + hcl_str(contract["endpoint_secret"]) + "\n"
        "    api_key_secret_name  = " + hcl_str(contract["apikey_secret"]) + "\n"
        "    policy_xml           = <<POLICY_XML\n"
        + policy_xml + "\n"
        "POLICY_XML\n"
        "  }\n"
        "]\n\n"
        "product_terms = " + hcl_str(product_terms) + "\n"
        + foundry_block
    )
    return content


for i, contract in enumerate(access_contracts, 1):
    utils.print_info(f"\n{'='*60}")
    utils.print_info(f"Generating tfvars {i}/{len(access_contracts)}: {contract['description']}")
    utils.print_info(f"{'='*60}")

    # Stable workspace + file name from use-case identity (no timestamp -> idempotent re-runs)
    ws = f"{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}".lower()
    tfvars_file = os.path.join(tfvars_out_dir, f"{ws}.tfvars")

    # Generate per-contract policy XML (model RBAC + capacity)
    policy_xml = build_product_policy_xml(contract)

    tfvars_content = build_tfvars(contract, policy_xml)
    with open(tfvars_file, "w") as f:
        f.write(tfvars_content)

    generated_tfvars_files.append({
        "contract": contract,
        "tfvars_file": tfvars_file,
        "workspace": ws,
    })
    utils.print_ok(f"✅ tfvars written: {tfvars_file}")
    utils.print_info(f"     • Workspace      : {ws}")
    utils.print_info(f"     • Key Vault      : {contract['use_keyvault']}  |  Foundry: {contract['use_foundry']}")
    utils.print_info(f"     • Allowed models : {', '.join(contract['allowed_models'])}")
    utils.print_info(f"     • TPM            : {contract['tokens_per_minute']}")
    utils.print_info(f"     • Token quota    : {contract['token_quota']} / {contract['token_quota_period']}")

utils.print_ok(f"\n📁 Generated {len(generated_tfvars_files)} Terraform tfvars files in {tfvars_out_dir}")
utils.print_info("Each contract is deployed into its own Terraform workspace for isolated state.")


<a id='4.1'></a>
### 4️⃣.1 Deploy Access Contracts Using 🌍 Terraform

Deploy each access contract using the generated `.tfvars` files against the standalone
`citadel-access-contracts` Terraform module. Each contract runs in its **own Terraform workspace**
(isolated state, no cross-clobber). The deploy step selects/creates the workspace, imports any
pre-existing APIM resources for idempotency, then runs `terraform apply`. This creates the APIM
products, subscriptions, and optionally Key Vault secrets and Foundry connections in Azure.

In [ ]:
# Deploy each generated access contract via the module's deploy.sh script.
# A dedicated Terraform workspace per contract keeps state isolated.
# deploy.sh handles: terraform init -upgrade -> import-existing.sh -> plan -> apply.
deployment_results = []

for i, item in enumerate(generated_tfvars_files, 1):
    contract = item["contract"]
    tfvars_file = item["tfvars_file"]
    ws = item["workspace"]

    utils.print_info(f"\n{'='*60}")
    utils.print_info(f"Deploying Access Contract {i}/{len(generated_tfvars_files)}: {contract['description']}")
    utils.print_info(f"Workspace: {ws}")
    utils.print_info(f"{'='*60}")

    # Select (or create) an isolated workspace, then delegate to deploy.sh.
    deploy_cmd = (
        f"cd '{tf_dir}' && "
        f"(terraform workspace select '{ws}' 2>/dev/null || terraform workspace new '{ws}') && "
        f"bash scripts/deploy.sh --auto-approve --var-file '{tfvars_file}'"
    )

    output = utils.run(
        deploy_cmd,
        f"Deployment '{ws}' succeeded",
        f"Deployment '{ws}' failed",
    )

    outputs = {}
    if output.success:
        # Capture Terraform outputs (full `terraform output -json` structure)
        tf_out = utils.run(
            f"cd '{tf_dir}' && terraform output -json",
            "Retrieved Terraform outputs",
            "Could not retrieve Terraform outputs",
        )
        if tf_out.success and tf_out.json_data:
            outputs = tf_out.json_data
            utils.print_ok(f"✅ Access Contract {i} deployed successfully!")
            for key, meta in outputs.items():
                masked_value = utils.mask_sensitive_values(meta.get("value"))
                utils.print_info(f"  {key}: {masked_value}")
        else:
            utils.print_ok(f"✅ Access Contract {i} deployed successfully!")
            utils.print_info("  (No outputs returned)")
    else:
        utils.print_error(f"❌ Access Contract {i} deployment failed!")

    deployment_results.append({
        "contract": contract,
        "workspace": ws,
        "outputs": outputs,
        "success": output.success,
    })

# Re-initialize APIM client to pick up new subscriptions
apimClientTool.initialize()
utils.print_ok(f"\n🎉 Completed deploying {len([r for r in deployment_results if r['success']])} access contracts!")


<a id='4.2'></a>
### 4️⃣.2 Foundry Connection Names (for downstream notebooks)

When `use_foundry = True` for an access contract, the deployment provisions a Microsoft Foundry
project connection that points back to the APIM gateway. **This connection name is what other
notebooks (e.g., `citadel-agent-frameworks-tests.ipynb`) use as `foundry_connection_name`.**

**Naming convention** (see `citadel-access-contracts/main.tf`):

```
<connection_name_prefix>-<ServiceCode>
```

Where `connection_name_prefix` defaults to `Hub-<business_unit>-<use_case_name>-<environment>` when left
empty in `foundry_config` (which is the default in this notebook).

So the resulting connection names follow the pattern:

```
Hub-<BusinessUnit>-<UseCaseName>-<Environment>-<ServiceCode>      # e.g. Hub-HR-ChatAgent-DEV-LLM
```

The next cell extracts the connection name(s) from the deployment outputs and prints them
explicitly so you can copy them into other notebooks.


In [ ]:
# Extract and display the Foundry connection name(s) for each deployed access contract.
# Use these values as `foundry_connection_name` in the agent-frameworks notebook.

foundry_connection_names = {}

print("\n" + "=" * 70)
print("🔗 FOUNDRY CONNECTION NAMES (for downstream notebooks)")
print("=" * 70)
print("Naming convention: Hub-<BusinessUnit>-<UseCaseName>-<Environment>-<ServiceCode>")
print("                   (when foundry_config.connection_name_prefix is empty)")
print("-" * 70)

for result in deployment_results:
    contract = result["contract"]
    if not contract.get("use_foundry"):
        continue
    if not result.get("success"):
        utils.print_warning(f"⚠️  Skipping {contract['name']} (deployment failed)")
        continue

    # Prefer the value emitted by the Terraform `foundry_connections` output
    # (a map keyed by service code -> { connection_name, connection_id, ... }).
    outputs = result.get("outputs") or {}
    fc_output = outputs.get("foundry_connections", {}).get("value", {})
    if fc_output:
        for entry in fc_output.values():
            conn_name = entry.get("connection_name") if isinstance(entry, dict) else entry
            if conn_name:
                foundry_connection_names.setdefault(contract["name"], []).append(conn_name)
    else:
        # Fallback: reconstruct from the naming convention
        for svc_code in ["LLM"]:
            conn_name = f"Hub-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}-{svc_code}"
            foundry_connection_names.setdefault(contract["name"], []).append(conn_name)

    for cn in foundry_connection_names.get(contract["name"], []):
        utils.print_ok(f"✅ {contract['description']}")
        print(f"     foundry_connection_name = \"{cn}\"")

if not foundry_connection_names:
    utils.print_info("No Foundry-enabled contracts in this run.")
else:
    print("-" * 70)
    print("📋 Copy/paste-ready map:")
    print(json.dumps(foundry_connection_names, indent=2))
print("=" * 70)


<a id='5'></a>
### 5️⃣ Retrieve API Keys for Each Access Contract

Get the subscription keys created for each access contract to use in API testing.

In [ ]:
# Map contract names to their subscription keys
contract_keys = {}

for result in deployment_results:
    if not result['success']:
        continue

    contract = result['contract']
    product_id = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
    subscription_name = f"{product_id}-SUB-01"

    # Find the subscription key from APIM subscriptions (the Terraform module sets a
    # deterministic subscription name matching `<code>-<postfix>-SUB-01`).
    for sub in apimClientTool.apim_subscriptions:
        if subscription_name.lower() in sub.get('name', '').lower():
            contract_keys[product_id] = {
                "key": sub.get('key'),
                "description": contract['description'],
                "use_keyvault": contract['use_keyvault'],
                "use_foundry": contract['use_foundry']
            }
            utils.print_ok(f"Found key for {product_id}")
            break
    else:
        # Fallback: for non-Key-Vault contracts, read the key directly from the
        # Terraform `endpoints` output (a map keyed by service code).
        if not contract['use_keyvault']:
            endpoints = (result['outputs'].get('endpoints', {}) or {}).get('value', {}) or {}
            ep = endpoints.get('LLM')
            if ep:
                contract_keys[product_id] = {
                    "key": ep.get('api_key'),
                    "endpoint": ep.get('endpoint'),
                    "description": contract['description'],
                    "use_keyvault": contract['use_keyvault'],
                    "use_foundry": contract['use_foundry']
                }
                utils.print_ok(f"Found direct key for {product_id}")

utils.print_info(f"\nRetrieved keys for {len(contract_keys)} access contracts:")
for product_id, info in contract_keys.items():
    utils.print_info(f"  • {product_id}: {info['description']}")


<a id='6'></a>
### 6️⃣ Test API Requests Across All Access Contracts

Send test requests to each access contract and collect metrics for visualization.

In [ ]:
model_name = "gpt-4.1-mini"
# if you want to dynamically select model from supported models, uncomment below
# model_name = supported_models[2] if len(supported_models) > 2 else supported_models[0]
utils.print_info(f"Using model: {model_name}")

# Store results for each contract
test_results = {product_id: [] for product_id in contract_keys.keys()}

messages = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is 2+2?"}
    ]
}

# Send a single test request to each contract
for product_id, info in contract_keys.items():
    utils.print_info(f"\nTesting {product_id}...")
    
    api_key = info.get('key')
    if not api_key:
        utils.print_error(f"No API key found for {product_id}")
        continue
    
    try:
        response = requests.post(
            chat_completions_url,
            headers={'api-key': api_key},
            json=messages,
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = json.loads(response.text)
            content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            utils.print_ok(f"💬 Response: {content[:100]}..." if len(content) > 100 else f"💬 Response: {content}")
            utils.print_info(f"   Region: {response.headers.get('x-ms-region', 'N/A')}")
        else:
            utils.print_error(f"Error: {response.text[:200]}")
    except Exception as e:
        utils.print_error(f"Request failed: {e}")

<a id='7'></a>
### 7️⃣ Run Load Test Across All Access Contracts

Send multiple requests to each contract over 30 seconds to test rate limiting and collect performance data.

In [ ]:
import requests, json, time
from concurrent.futures import ThreadPoolExecutor
import threading

# Run for 30 seconds per contract
test_duration = 30
all_api_runs = {product_id: [] for product_id in contract_keys.keys()}

model_name = "gpt-4.1" # This is a common model that is allowed by default in the contracts, but you can change it as needed
utils.print_info(f"Using model: {model_name}")

messages = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Count from 1 to 10."}
    ]
}

def run_api_test(product_id, api_key, duration):
    """Run API calls for a specific contract."""
    runs = []
    start_time = time.time()
    run_count = 0
    
    while (time.time() - start_time) < duration:
        run_count += 1
        call_start_time = time.time()
        
        try:
            response = requests.post(
                chat_completions_url,
                headers={'api-key': api_key},
                json=messages,
                timeout=30
            )
            
            elapsed = time.time() - start_time
            
            if response.status_code == 200:
                data = json.loads(response.text)
                total_tokens = data.get("usage", {}).get("total_tokens", 0)
            else:
                total_tokens = 0
            
            runs.append((call_start_time, total_tokens, response.status_code, elapsed))
            
        except Exception as e:
            runs.append((call_start_time, 0, 500, time.time() - start_time))
        
        time.sleep(0.2)  # Small delay between requests
    
    return runs

# Run tests for each contract sequentially
for product_id, info in contract_keys.items():
    api_key = info.get('key')
    if not api_key:
        continue
    
    print(f"\n🕐 Testing {product_id} for {test_duration} seconds...")
    print(f"   {info['description']}")
    
    runs = run_api_test(product_id, api_key, test_duration)
    all_api_runs[product_id] = runs
    
    success = sum(1 for r in runs if r[2] == 200)
    throttled = sum(1 for r in runs if r[2] == 429)
    errors = sum(1 for r in runs if r[2] not in [200, 429])
    
    print(f"   ✅ Success: {success} | ⛔ Throttled: {throttled} | ❌ Errors: {errors}")

utils.print_ok(f"\n🏁 Load testing completed for all {len(contract_keys)} access contracts!")

<a id='8'></a>
### 8️⃣ Visualize Results Across All Access Contracts

Compare API usage, token consumption, and throttling behavior across all access contracts.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

# Check if we have data to plot
contracts_with_data = {k: v for k, v in all_api_runs.items() if v}

if contracts_with_data:
    # Print summary table
    print("\n" + "="*80)
    print("📊 SUMMARY: Access Contracts Performance Comparison")
    print("="*80)
    print(f"{'Contract':<40} {'Calls':<8} {'Success':<10} {'Throttled':<10} {'Tokens':<10}")
    print("-"*80)
    
    for product_id, runs in contracts_with_data.items():
        success = sum(1 for r in runs if r[2] == 200)
        throttled = sum(1 for r in runs if r[2] == 429)
        total_tokens = sum(r[1] for r in runs)
        print(f"{product_id:<40} {len(runs):<8} {success:<10} {throttled:<10} {total_tokens:<10}")
    
    print("="*80)
    
    num_contracts = len(contracts_with_data)
    fig, axes = plt.subplots(num_contracts, 1, figsize=(14, 5 * num_contracts), squeeze=False)
    
    colors_map = {'success': 'tab:green', 'throttled': 'tab:red', 'error': 'tab:orange'}
    
    for idx, (product_id, runs) in enumerate(contracts_with_data.items()):
        ax = axes[idx, 0]
        
        if not runs:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.set_title(product_id)
            continue
        
        # Process data
        base_time = runs[0][0]
        times = [r[3] for r in runs]  # elapsed time
        tokens = [r[1] for r in runs]
        status_codes = [r[2] for r in runs]
        
        # Color bars based on status
        colors = [
            colors_map['success'] if code == 200 
            else colors_map['throttled'] if code == 429 
            else colors_map['error'] 
            for code in status_codes
        ]
        
        # Create bar chart
        ax.bar(times, tokens, color=colors, width=0.3, alpha=0.7)
        
        # Add throttled markers
        throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
        if throttled_times:
            max_tokens = max(tokens) if tokens else 1
            ax.scatter(throttled_times, [max_tokens * 0.05] * len(throttled_times), 
                      marker='x', s=50, color='darkred', zorder=5)
        
        # Calculate stats
        success = sum(1 for code in status_codes if code == 200)
        throttled = sum(1 for code in status_codes if code == 429)
        total_tokens = sum(tokens)
        
        # Labels and title
        info = contract_keys.get(product_id, {})
        title = f"{product_id}\n{info.get('description', '')}"
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Tokens per call')
        
        # Add stats annotation
        stats_text = f"Total: {len(runs)} calls | Success: {success} | Throttled: {throttled} | Tokens: {total_tokens}"
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Add shared legend
    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Patch(facecolor='tab:red', alpha=0.7, label='Throttled (429)'),
        Patch(facecolor='tab:orange', alpha=0.7, label='Error'),
        Line2D([0], [0], marker='x', color='darkred', markersize=8, linestyle='None', label='Throttle point')
    ]
    fig.legend(handles=legend_items, loc='upper right', bbox_to_anchor=(0.98, 0.99))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    plt.show()
    
else:
    print('No API test data available. Run the load test first to capture data.')

<a id='9'></a>
### 9️⃣ Compare Token Bucket Behavior

Visualize the token bucket algorithm behavior for each access contract.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

contracts_with_data = {k: v for k, v in all_api_runs.items() if v}

if contracts_with_data:
    # Token bucket parameters (default policy: 400 tokens/min)
    capacity = 400
    refill = capacity / 60  # tokens per second
    
    fig, axes = plt.subplots(len(contracts_with_data), 1, figsize=(14, 6 * len(contracts_with_data)), squeeze=False)
    
    for idx, (product_id, runs) in enumerate(contracts_with_data.items()):
        ax1 = axes[idx, 0]
        ax2 = ax1.twinx()
        
        # Process data for token bucket simulation
        calls = [(r[3], r[1] or 0, r[2]) for r in runs]  # (elapsed_time, tokens, status)
        
        bucket = capacity
        last_time = 0.0
        times, usage, status_codes, levels = [], [], [], []
        
        for call_time, tokens, status in calls:
            # Refill bucket
            bucket = min(capacity, bucket + (call_time - last_time) * refill)
            levels.append(bucket)
            times.append(call_time)
            usage.append(tokens)
            status_codes.append(status)
            # Consume tokens
            bucket = max(0, bucket - tokens)
            last_time = call_time
        
        # Colors based on status
        colors = ['tab:green' if code == 200 else 'tab:red' if code == 429 else 'tab:orange' for code in status_codes]
        
        # Plot bars for token usage
        ax1.bar(times, usage, color=colors, width=0.35, alpha=0.7)
        
        # Plot bucket level
        ax2.plot(times, levels, color='purple', linewidth=2)
        ax2.axhline(capacity, color='purple', linestyle='--', alpha=0.6)
        
        # Mark throttled points
        throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
        throttled_usage = [u for u, code in zip(usage, status_codes) if code == 429]
        if throttled_times:
            max_usage = max(usage) if usage else 0
            throttled_marker_heights = [u + max_usage * 0.01 for u in throttled_usage]
            ax1.scatter(throttled_times, throttled_marker_heights, marker='o', s=20, 
                       color='darkred', edgecolors='white', linewidth=0.4, zorder=6)
        
        # Labels
        ax1.set_xlabel('Seconds')
        ax1.set_ylabel('Tokens per call')
        ax2.set_ylabel('Tokens in bucket', color='purple')
        ax2.tick_params(axis='y', labelcolor='purple')
        
        info = contract_keys.get(product_id, {})
        ax1.set_title(f'Token Bucket Behavior: {product_id}\n{info.get("description", "")}')
        
        # Stats
        success = sum(code == 200 for code in status_codes)
        throttled = sum(code == 429 for code in status_codes)
        print(f"{product_id}: Calls: {len(status_codes)} | Success: {success} | Throttled: {throttled}")
    
    # Add legend to first subplot
    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Line2D([0], [0], color='purple', linewidth=2, label='Bucket level'),
        Line2D([0], [0], color='purple', linestyle='--', label='Capacity'),
        Line2D([0], [0], marker='o', color='darkred', markersize=8, linestyle='None',
               markerfacecolor='darkred', markeredgecolor='white', label='Throttled (429)')
    ]
    axes[0, 0].legend(handles=legend_items, loc='upper right', bbox_to_anchor=(0.98, 0.85), framealpha=0.9)
    
    plt.tight_layout()
    plt.show()
else:
    print('Run the load test first to capture api_runs data.')

---
## 📊 Results Summary
---

In [ ]:
# Overall summary
utils.print_info(f"\n{'='*80}")
utils.print_info(f"📊 ACCESS CONTRACTS TEST SUMMARY")
utils.print_info(f"{'='*80}")

utils.print_info(f"\n📜 Access Contracts Tested: {len(contract_keys)}")
for product_id, info in contract_keys.items():
    utils.print_info(f"   • {product_id}: {info['description']}")

if contracts_with_data:
    total_calls = sum(len(runs) for runs in contracts_with_data.values())
    total_success = sum(sum(1 for r in runs if r[2] == 200) for runs in contracts_with_data.values())
    total_throttled = sum(sum(1 for r in runs if r[2] == 429) for runs in contracts_with_data.values())
    total_tokens_all = sum(sum(r[1] for r in runs) for runs in contracts_with_data.values())

    utils.print_info(f"\n📈 Load Test Results:")
    utils.print_info(f"   Total API calls: {total_calls}")
    utils.print_ok(f"   Successful: {total_success}")
    if total_throttled > 0:
        utils.print_info(f"   Throttled: {total_throttled}")
    utils.print_info(f"   Total tokens consumed: {total_tokens_all}")

    if total_success == total_calls:
        utils.print_ok(f"\n✅ All tests passed!")
    else:
        utils.print_info(f"\n📊 Pass rate: {total_success}/{total_calls}")
else:
    utils.print_info("\n⚠️ No load test data available. Run the load test cells first.")

<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the test access contracts from APIM created during this notebook session.

> **Note:** This will not delete any created secrets in Azure Key Vault or Microsoft Foundry connections.

In [ ]:
# Set to True to delete the access contracts created in this session
cleanup_enabled = False

if cleanup_enabled:
    for result in deployment_results:
        if not result['success']:
            continue
        
        contract = result['contract']
        product_id = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
        subscription_name = f"{product_id}-SUB-01"
        
        utils.print_info(f"Deleting {product_id}...")
        
        # Delete product and its associated subscriptions
        prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {product_id} --delete-subscriptions true --yes"
        utils.run(prod_cmd, f"Deleted product {product_id}", f"Failed to delete product")
    
    utils.print_ok("Cleanup completed!")
else:
    utils.print_info("Cleanup is disabled. Set cleanup_enabled = True to remove test resources.")